# E01 Iris（Fisher 1936）：多分类、决策边界与分层验证

> Issue #9 | [UCI Iris](https://archive.ics.uci.edu/ml/datasets/iris) | Fisher (1936) 线性判别分析经典数据

## 你将学到
1. **分类 vs 回归**：分类预测离散标签，回归预测连续量；决策区域由分类器在特征空间划分的区域决定（见 [sklearn 分类概览](https://scikit-learn.org/stable/supervised_learning.html#classification)）
2. **可线性分隔性**：*Iris setosa* 与其余两类几乎线性可分；*versicolor* 与 *virginica* 在花瓣特征上有重叠
3. **小样本多分类**：每类仅 50 条 → 必须 **分层** 划分与 CV，否则某折缺类、指标方差膨胀
4. **预处理泄漏**：`StandardScaler` 只能在训练折上 `fit`；用 `Pipeline` 把缩放与模型绑定
5. **提交格式**：整数 `target` vs 字符串 `species`；`prediction_type` 用 `label` / `proba` / `auto`（**禁止** `probability`）

## 技术难点
- 花瓣长度/宽度对判别贡献最大（萼片特征次要）
- 混淆矩阵读法：哪两类互混？macro-F1 对类不均衡更稳健


In [ ]:
QUICK = True
RANDOM_STATE = 42

# 一键就绪：关警告 + 打印环境 + 设种子（勿再叠 JE.no_warning / filterwarnings）
from wanglaoshi.runtime import ready
ready(seed=RANDOM_STATE)


## 0. 分类与回归：任务类型一览


In [ ]:
# 本 Notebook 是 **多分类**；若预测连续花瓣长度则是回归。
# 分类器学习 f: R^4 -> {0,1,2}，等价于在 R^4 中划出三个决策区域（decision regions）。
TASK = 'multiclass_classification'
print('task =', TASK)


## 1. 加载 Iris：数字标签 + 物种类名字符串


In [ ]:
from sklearn.datasets import load_iris
import pandas as pd
import numpy as np

iris = load_iris(as_frame=True)
df = iris.frame.copy()
df['species'] = iris.target_names[iris.target]
feature_cols = [c for c in df.columns if c not in ('target', 'species')]
print('shape =', df.shape)
print('features:', feature_cols)
print('target counts:\n', df['target'].value_counts().sort_index())
print('species counts:\n', df['species'].value_counts())
df.head()


## 2. EDA：描述统计与缺失


In [ ]:
from wanglaoshi.analyze import DataAnalyzer

analyzer = DataAnalyzer(df)
print(analyzer.get_basic_info())
display(analyzer.basic_statistics())
display(analyzer.missing_value_analysis())


### 2.1 相关结构：花瓣特征主导判别

Pearson 相关系数衡量线性关联；与 `target` 相关性高的特征往往在 PCA/逻辑回归中权重大。


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

corr = df[feature_cols + ['target']].corr()
plt.figure(figsize=(5, 4))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0)
plt.title('Feature correlation (incl. target)')
plt.tight_layout()
plt.show()
petal_corr = corr['target'].drop('target').abs().sort_values(ascending=False)
print('|corr(target)| ranking:\n', petal_corr)


### 2.2 配对图：Setosa 分离 vs Versicolor/Virginica 重叠


In [ ]:
# QUICK 下用全量 150 条仍可接受
g = sns.pairplot(
    df, hue='species', vars=feature_cols,
    corner=True, plot_kws={'s': 25, 'alpha': 0.8}, height=2.0
)
g.fig.suptitle('Pairplot by species', y=1.02)
plt.show()


### 2.3 二维切片：Petal length × Petal width


In [ ]:
plt.figure(figsize=(6, 4.5))
for sp, mk in zip(iris.target_names, ('o', 's', '^')):
    sub = df[df['species'] == sp]
    plt.scatter(sub['petal length (cm)'], sub['petal width (cm)'], label=sp, marker=mk, s=40, alpha=0.85)
plt.xlabel('petal length (cm)')
plt.ylabel('petal width (cm)')
plt.legend()
plt.title('Decision-boundary intuition: setosa isolated on the left')
plt.grid(True, alpha=0.3)
plt.show()


## 3. 分层划分：为什么小样本多分类必须 stratify？

若随机划分，验证集可能某一类过少甚至为 0，Accuracy / F1 会剧烈波动。
下面用 **类计数** 与 **CV 标准差** 对比分层 vs 非分层（后者仅作反面教材）。


In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, KFold

X = df[feature_cols]
y_int = df['target']
y_str = df['species']

Xtr, Xte, ytr, yte = train_test_split(
    X, y_int, test_size=0.3, random_state=RANDOM_STATE, stratify=y_int
)
print('✅ stratified train:', ytr.value_counts().sort_index().to_dict())
print('✅ stratified test :', yte.value_counts().sort_index().to_dict())

# 反面教材：固定 seed=7 且无 stratify（不用于正式评估）
_, _, ytr_bad, yte_bad = train_test_split(
    X, y_int, test_size=0.3, random_state=7, stratify=None
)
print('❌ non-strat train:', ytr_bad.value_counts().sort_index().to_dict())
print('❌ non-strat test :', yte_bad.value_counts().sort_index().to_dict())


### 3.1 交叉验证：分层 K 折 vs 普通 K 折的稳定性


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
n_splits = 3 if QUICK else 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
kf = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
acc_strat = cross_val_score(pipe, X, y_int, cv=skf, scoring='accuracy')
acc_plain = cross_val_score(pipe, X, y_int, cv=kf, scoring='accuracy')
print(f'StratifiedKFold accuracy: {acc_strat.mean():.3f} ± {acc_strat.std():.3f}')
print(f'KFold (no strat) accuracy: {acc_plain.mean():.3f} ± {acc_plain.std():.3f}')
print('通常分层 CV 的 std 更小 —— 更可信的泛化估计')


## 4. 数据泄漏示范：在全量数据上 fit 缩放器

**错误**：`scaler.fit(X)` 用到了验证/测试集的均值方差 → 乐观偏差。  
**正确**：只在 `Xtr` 上 `fit`，或把 `StandardScaler` 放进 `Pipeline` 并在 CV 各折内 fit。


In [ ]:
from sklearn.metrics import accuracy_score

# ❌ 泄漏：在全量 X 上 fit
from sklearn.preprocessing import StandardScaler as SS
bad_scaler = SS()
X_all_scaled = bad_scaler.fit_transform(X)
Xtr_bad, Xte_bad, _, _ = train_test_split(
    X_all_scaled, y_int, test_size=0.3, random_state=RANDOM_STATE, stratify=y_int
)
clf_bad = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
clf_bad.fit(Xtr_bad, ytr)
acc_leak = accuracy_score(yte, clf_bad.predict(Xte_bad))

# ✅ Pipeline：缩放仅在训练折 fit
clf_good = Pipeline([
    ('scaler', SS()),
    ('clf', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
clf_good.fit(Xtr, ytr)
acc_pipe = accuracy_score(yte, clf_good.predict(Xte))
print(f'accuracy with leaky scaling: {acc_leak:.3f}')
print(f'accuracy with Pipeline scaling: {acc_pipe:.3f}')


## 5. QuickBaseline：多模型对比（训练集上 fit，验证集评估）


In [ ]:
from wanglaoshi import compete as CT

baseline = CT.QuickBaseline()
results = baseline.run_all_models(Xtr, ytr, Xte, yte, task_type='classification')
comparison = baseline.compare_models(results, top_n=6)
comparison


## 6. 混淆矩阵与 macro-F1 解读


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score

best = comparison.iloc[0]['Model']
pred_int = np.asarray(results[best]['predictions']).astype(int)
print(classification_report(yte, pred_int, target_names=list(iris.target_names)))
cm = confusion_matrix(yte, pred_int)
print('confusion matrix (rows=true, cols=pred):\n', cm)
macro_f1 = f1_score(yte, pred_int, average='macro')
print('macro-F1 =', round(macro_f1, 4))
print('macro-F1 对每一类 F1 等权平均 —— 小类不被大类 Accuracy 淹没')


### 6.1 PCA 二维可视化：类别在主轴上的分离


In [ ]:
from sklearn.decomposition import PCA

pca_vis = PCA(n_components=2, random_state=RANDOM_STATE)
Z = pca_vis.fit_transform(Xtr)
plt.figure(figsize=(6, 5))
sc = plt.scatter(Z[:, 0], Z[:, 1], c=ytr, cmap='tab10', s=35, edgecolors='k', linewidths=0.3)
plt.colorbar(sc, ticks=[0, 1, 2], label='target')
plt.xlabel(f'PC1 ({pca_vis.explained_variance_ratio_[0]:.1%} var)')
plt.ylabel(f'PC2 ({pca_vis.explained_variance_ratio_[1]:.1%} var)')
plt.title('PCA 2D on train (fit on train only)')
plt.grid(True, alpha=0.3)
plt.show()
print('累计解释方差:', pca_vis.explained_variance_ratio_.sum())


## 7. 提交：整数标签 vs 字符串物种名


In [ ]:
from pathlib import Path
from wanglaoshi import compete as CT

out = Path('outputs/01_iris')
out.mkdir(parents=True, exist_ok=True)
gen = CT.SubmissionGenerator()

# A) 整数 target
sample_int = pd.DataFrame({'Id': np.arange(len(pred_int)), 'target': yte.to_numpy()})
sample_int.to_csv(out / 'sample_int.csv', index=False)
sub_int = gen.create_submission(
    pred_int, str(out / 'sample_int.csv'), str(out / 'sub_int.csv'), prediction_type='label'
)
print('INT validate:', gen.validate_submission(str(out / 'sub_int.csv'), str(out / 'sample_int.csv')))

# B) 字符串 species
pred_str = iris.target_names[pred_int]
sample_str = pd.DataFrame({
    'Id': np.arange(len(pred_str)),
    'species': y_str.loc[yte.index].to_numpy(),
})
sample_str.to_csv(out / 'sample_str.csv', index=False)
sub_str = gen.create_submission(
    pred_str, str(out / 'sample_str.csv'), str(out / 'sub_str.csv'),
    prediction_type='label', target_column='species',
)
print('STR validate:', gen.validate_submission(str(out / 'sub_str.csv'), str(out / 'sample_str.csv')))
display(sub_int.head(3))
display(sub_str.head(3))


### 7.1 若需概率提交：使用 `prediction_type='proba'`


In [ ]:
# 禁止 prediction_type='probability' —— 库枚举为 auto | label | proba | multioutput
model = results[best]['model']
if hasattr(model, 'predict_proba'):
    proba = model.predict_proba(Xte)
    print('proba shape', proba.shape, '— 竞赛多列概率时用 prediction_type=\'proba\'')
else:
    print('当前最优模型无 predict_proba，本单元跳过')


### 7.2 二类子问题：Versicolor vs Virginica（最难边界）


In [ ]:
mask_two = df['species'].isin(['versicolor', 'virginica'])
X2 = df.loc[mask_two, feature_cols]
y2 = df.loc[mask_two, 'target']
X2_tr, X2_te, y2_tr, y2_te = train_test_split(
    X2, y2, test_size=0.3, random_state=RANDOM_STATE, stratify=y2
)
pipe2 = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
])
pipe2.fit(X2_tr, y2_tr)
pred2 = pipe2.predict(X2_te)
print('binary versicolor/virginica only:')
print(classification_report(y2_te, pred2, target_names=['versicolor', 'virginica']))
print('macro-F1', f1_score(y2_te, pred2, average='macro'))


### 7.3 二维决策区域（仅花瓣两维，教学可视化）


In [ ]:
from matplotlib.colors import ListedColormap

pair = ['petal length (cm)', 'petal width (cm)']
X_vis = df[pair].to_numpy()
y_vis = df['target'].to_numpy()
Xv_tr, Xv_te, yv_tr, yv_te = train_test_split(
    X_vis, y_vis, test_size=0.3, random_state=RANDOM_STATE, stratify=y_vis
)
clf_vis = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
clf_vis.fit(Xv_tr, yv_tr)
x_min, x_max = X_vis[:, 0].min() - 0.5, X_vis[:, 0].max() + 0.5
y_min, y_max = X_vis[:, 1].min() - 0.5, X_vis[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 120), np.linspace(y_min, y_max, 120))
Zmesh = clf_vis.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
plt.figure(figsize=(6, 5))
plt.contourf(xx, yy, Zmesh, alpha=0.25, cmap=ListedColormap(['#FFAAAA', '#AAFFAA', '#AAAAFF']))
plt.scatter(Xv_te[:, 0], Xv_te[:, 1], c=yv_te, cmap=ListedColormap(['r', 'g', 'b']), edgecolors='k', s=45)
plt.xlabel(pair[0])
plt.ylabel(pair[1])
plt.title('Decision regions (logistic, petal features only)')
plt.show()


### 7.4 混淆矩阵热力图与 per-class F1


In [ ]:
import seaborn as sns

plt.figure(figsize=(4.5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=iris.target_names, yticklabels=iris.target_names)
plt.xlabel('predicted')
plt.ylabel('true')
plt.title('Confusion matrix heatmap')
plt.tight_layout()
plt.show()
per_f1 = f1_score(yte, pred_int, average=None)
for name, f1v in zip(iris.target_names, per_f1):
    print(f'F1[{name}] = {f1v:.3f}')


### 7.5 重复分层 CV：accuracy / macro-F1 双指标


In [ ]:
from sklearn.model_selection import cross_validate

scoring = {'acc': 'accuracy', 'f1_macro': 'f1_macro'}
cv_out = cross_validate(pipe, X, y_int, cv=skf, scoring=scoring)
print('CV accuracy', cv_out['test_acc'].mean(), '±', cv_out['test_acc'].std())
print('CV macro-F1', cv_out['test_f1_macro'].mean(), '±', cv_out['test_f1_macro'].std())


### 7.6 Fisher LDA 历史注记

Fisher (1936) 在线性判别框架下用 Iris 说明类间可分性与投影方向；现代 `LogisticRegression` / SVM 是同一决策几何的不同估计方式。参见 [Fisher Iris 原论文摘要](https://aspace.agilis-network.com/handle/20.500.12142/2007)。


In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

lda = Pipeline([
    ('scaler', StandardScaler()),
    ('lda', LinearDiscriminantAnalysis()),
])
lda.fit(Xtr, ytr)
lda_acc = lda.score(Xte, yte)
print('LDA test accuracy:', lda_acc)
coef = lda.named_steps['lda'].coef_
print('LDA coef shape', coef.shape, '— each row is class vs rest direction')


## 8. 检查清单
- [x] 分类任务与决策区域直觉
- [x] 花瓣特征主导 + Setosa 可分
- [x] 分层划分 vs 非分层 + CV std 对比
- [x] Pipeline 避免缩放泄漏
- [x] 混淆矩阵 + macro-F1
- [x] PCA 2D 类别可视化
- [x] 整数/字符串 SubmissionGenerator
- [ ] 对 versicolor/virginica 专门做二元分类实验


## 附录 A：判别分析直觉（与 Fisher 原文的联系）

Fisher (1936) 用线性组合最大化「类间方差 / 类内方差」。现代 Logistic / SVM 是不同损失函数下的分界学习，但 **几何直觉相同**：先看类是否可分，再选模型复杂度。

课堂练习：若只用 `petal length` 一个特征做阈值分类，Setosa 是否仍可完美分开？下面用单特征直方图验证。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, col in zip(axes, ['petal length (cm)', 'sepal length (cm)']):
    for sp, g in df.groupby('species'):
        ax.hist(g[col], bins=12, alpha=0.45, label=sp)
    ax.set_title(col); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()
print('结论：单靠 petal length 已能清开 Setosa；sepal length 三类重叠更多。')


## 附录 B：为何宏平均 F1 适合本课

Accuracy 在均衡三类上尚可，但一旦你改成「稀有类检测」任务思维，就必须看每类召回。宏平均对三类一视同仁：

\[\mathrm{macro\text{-}F1}=\frac{1}{K}\sum_{k=1}^{K} F1_k\]

作业：把验证集人为丢掉一半 virginica，再报 Accuracy 与 macro-F1，体会两者分歧。


In [ ]:
from sklearn.metrics import accuracy_score, f1_score
import numpy as np
# 统一取预测：兼容不同课脚本变量名
if "pred" not in globals():
    if "pred_int" in globals():
        pred = pred_int
    elif "pred2" in globals():
        pred = pred2
    elif "results" in globals() and "comparison" in globals():
        best = comparison.iloc[0]["Model"]
        pred = results[best]["predictions"]
    else:
        raise NameError("no predictions found")
mask = (yte != 2) | (np.arange(len(yte)) % 2 == 0)
yte_imbalanced = yte[mask]
pred_arr = np.asarray(pred)
pred_sub = pred_arr[np.asarray(mask)]
print("subset size", len(yte_imbalanced), "virginica", int((yte_imbalanced==2).sum()))
print("acc", accuracy_score(yte_imbalanced, pred_sub))
print("macro-F1", f1_score(yte_imbalanced, pred_sub, average="macro"))

